In [5]:
# ============================================================
# SMB Silver Transform
# ============================================================

# Imports + Load Bronze

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "bronze_smb_accounts"
SILVER_TABLE = "silver_smb_accounts"
QUARANTINE_TABLE = "silver_smb_accounts_quarantine"

df = spark.table(BRONZE_TABLE)

print(f"Bronze rows loaded: {df.count():,}")
print(f"Bronze columns    : {len(df.columns)}")
df.printSchema()
display(df.limit(10))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 7, Finished, Available, Finished, False)

Bronze rows loaded: 60,000
Bronze columns    : 17
root
 |-- account_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- signup_month: string (nullable = true)
 |-- month: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- employee_count: string (nullable = true)
 |-- revenue_tier: string (nullable = true)
 |-- region: string (nullable = true)
 |-- m365_active_users: string (nullable = true)
 |-- m365_licensed_seats: string (nullable = true)
 |-- teams_daily_active: string (nullable = true)
 |-- azure_compute_hours: string (nullable = true)
 |-- dynamics_users: string (nullable = true)
 |-- security_product: string (nullable = true)
 |-- support_tickets: string (nullable = true)
 |-- estimated_clv_12mo: string (nullable = true)
 |-- churned: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 0f5fd718-d87b-495a-bdbb-c702363114c1)

In [6]:
# Baseline anomaly profiling

profile_df = df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("account_id").isNull().cast("int")).alias("null_account_id"),
    F.sum(F.col("signup_month").isNull().cast("int")).alias("null_signup_month"),
    F.sum(F.col("month").isNull().cast("int")).alias("null_month_raw"),
    F.sum(F.col("m365_active_users").isNull().cast("int")).alias("null_active_users"),
    F.sum(F.col("azure_compute_hours").isNull().cast("int")).alias("null_azure_hours"),
    F.sum((F.col("azure_compute_hours") < 0).cast("int")).alias("negative_azure_hours"),
    F.sum((F.col("support_tickets") == 99).cast("int")).alias("support_ticket_99"),
    F.sum(((F.col("m365_licensed_seats") < 0) | (F.col("m365_licensed_seats") == 9999)).cast("int")).alias("bad_licensed_seats"),
    F.sum((F.col("teams_daily_active") > F.col("m365_active_users")).cast("int")).alias("teams_gt_active"),
    F.sum((F.col("dynamics_users") > F.col("m365_active_users")).cast("int")).alias("dynamics_gt_active")
)

display(profile_df)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ac4a1de2-2851-42c1-ae13-9f2d7fcb695f)

In [7]:
# Standardize month and core fields

df = (
    df
    .withColumn("account_id", F.col("account_id").cast("int"))
    .withColumn("signup_month", F.col("signup_month").cast("int"))
    .withColumn("month_raw", F.col("month"))
    .withColumn("month", F.regexp_extract(F.col("month").cast("string"), r"\d+", 0).cast("int"))
    .withColumn("company_name", F.trim(F.col("company_name")))
)

display(df.select("month_raw", "month").limit(20))
display(df.select("month").distinct().orderBy("month"))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, edcf8a72-2c26-4deb-83be-92d425326abb)

SynapseWidget(Synapse.DataFrame, 7fc589fa-a697-42ed-b449-d966004c80a6)

In [8]:
# Standardize category columns

df = df.withColumn("industry_raw", F.col("industry"))
df = df.withColumn("revenue_tier_raw", F.col("revenue_tier"))

# industry_clean
df = df.withColumn("industry_clean", F.trim(F.lower(F.col("industry"))))
df = df.withColumn(
    "industry",
    F.when(F.col("industry_clean").like("%retail%"), "Retail")
     .when(F.col("industry_clean").like("%finance%"), "Finance")
     .when(F.col("industry_clean").like("%health%"), "Healthcare")
     .when(F.col("industry_clean").like("%manufacturing%"), "Manufacturing")
     .when(F.col("industry_clean").like("%tech%"), "Tech")
     .otherwise(None)
)

# revenue_tier
df = df.withColumn("revenue_tier_clean", F.trim(F.lower(F.col("revenue_tier"))))
df = df.withColumn(
    "revenue_tier",
    F.when(F.col("revenue_tier_clean").like("%low%"), "Low")
     .when(F.col("revenue_tier_clean").like("%mid%"), "Mid")
     .when(F.col("revenue_tier_clean").like("%high%"), "High")
     .otherwise(None)
)

display(df.select("industry_raw", "industry").distinct().orderBy("industry_raw"))
display(df.select("revenue_tier_raw", "revenue_tier").distinct().orderBy("revenue_tier_raw"))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7bab7fa-addf-4200-9ab9-b752bdf909b0)

SynapseWidget(Synapse.DataFrame, ab0b7281-fa57-49ea-84a8-85731e7eb8ce)

In [9]:
# Quarantine unrecoverable rows

quarantine_df = df.filter(
    F.col("account_id").isNull() |
    F.col("month").isNull() |
    (~F.col("month").between(1, 12)) |
    F.col("signup_month").isNull() |
    (~F.col("signup_month").between(1, 12)) |
    F.col("industry").isNull() |
    F.col("revenue_tier").isNull()
)

silver_df = df.filter(
    F.col("account_id").isNotNull() &
    F.col("month").between(1, 12) &
    F.col("signup_month").between(1, 12) &
    F.col("industry").isNotNull() &
    F.col("revenue_tier").isNotNull()
)

print(f"Quarantine rows: {quarantine_df.count():,}")
print(f"Rows retained  : {silver_df.count():,}")

quarantine_df.write.mode("overwrite").saveAsTable(QUARANTINE_TABLE)
print(f"Saved quarantine table: {QUARANTINE_TABLE}")

# Isolated unrecoverable records into a quarantine table and retained only analytically valid rows in Silver.

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 11, Finished, Available, Finished, False)

Quarantine rows: 1,500
Rows retained  : 58,500
Saved quarantine table: silver_smb_accounts_quarantine


In [10]:
# Add business-context flags
# is_pre_signup = 1 when the monthly snapshot happened before the account’s signup_month
silver_df = (
    silver_df
    .withColumn("is_pre_signup", F.when(F.col("month") < F.col("signup_month"), 1).otherwise(0))
    .withColumn(
        "snapshot_date",
        F.to_date(F.concat(F.lit("2025-"), F.lpad(F.col("month"), 2, "0"), F.lit("-01")))
    )
)

display(silver_df.select("account_id", "signup_month", "month", "is_pre_signup", "snapshot_date").limit(20))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6353163e-6ce2-4b48-983b-1e693293bc99)

In [11]:
# Convert sentinels / invalid values to null

silver_df = (
    silver_df
    .withColumn(
        "m365_licensed_seats",
        F.when(
            (F.col("m365_licensed_seats") < 0) | (F.col("m365_licensed_seats") == 9999),
            None
        ).otherwise(F.col("m365_licensed_seats").cast("int"))
    )
    .withColumn(
        "m365_active_users",
        F.col("m365_active_users").cast("int")
    )
    .withColumn(
        "teams_daily_active",
        F.col("teams_daily_active").cast("int")
    )
    .withColumn(
        "azure_compute_hours",
        F.when(
            F.col("azure_compute_hours").isNull() | (F.col("azure_compute_hours") < 0),
            None
        ).otherwise(F.col("azure_compute_hours").cast("double"))
    )
    .withColumn(
        "dynamics_users",
        F.col("dynamics_users").cast("int")
    )
    .withColumn(
        "security_product",
        F.col("security_product").cast("int")
    )
    .withColumn(
        "support_tickets",
        F.when(F.col("support_tickets") == 99, None)
         .otherwise(F.col("support_tickets").cast("int"))
    )
    .withColumn(
        "estimated_clv_12mo",
        F.col("estimated_clv_12mo").cast("double")
    )
    .withColumn(
        "churned",
        F.col("churned").cast("int")
    )
)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 13, Finished, Available, Finished, False)

In [12]:
# Business-aware null handling

silver_df = (
    silver_df
    .withColumn(
        "m365_licensed_seats",
        F.when(F.col("is_pre_signup") == 1, F.lit(0))
         .otherwise(F.coalesce(F.col("m365_licensed_seats"), F.lit(0)))
    )
    .withColumn(
        "m365_active_users",
        F.when(F.col("is_pre_signup") == 1, F.lit(0))
         .otherwise(F.coalesce(F.col("m365_active_users"), F.lit(0)))
    )
    .withColumn(
        "azure_compute_hours",
        F.when(F.col("is_pre_signup") == 1, F.lit(0.0))
         .otherwise(F.coalesce(F.col("azure_compute_hours"), F.lit(0.0)))
    )
    .withColumn(
        "support_tickets",
        F.when(F.col("is_pre_signup") == 1, F.lit(0))
         .otherwise(F.coalesce(F.col("support_tickets"), F.lit(0)))
    )
)

display(
    silver_df.select(
        F.sum(F.col("m365_licensed_seats").isNull().cast("int")).alias("licensed_nulls"),
        F.sum(F.col("m365_active_users").isNull().cast("int")).alias("active_nulls"),
        F.sum(F.col("azure_compute_hours").isNull().cast("int")).alias("azure_nulls"),
        F.sum(F.col("support_tickets").isNull().cast("int")).alias("support_nulls")
    )
)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d63376f5-a32f-4c20-a885-0d70f9c0eef2)

In [14]:
# Enforce logical constraints

silver_df = (
    silver_df
    .withColumn(
        "m365_licensed_seats",
        F.when(F.col("m365_licensed_seats") < 0, 0)
         .otherwise(F.col("m365_licensed_seats"))
    )
    .withColumn(
        "m365_active_users",
        F.when(F.col("m365_active_users") < 0, 0)
         .otherwise(F.col("m365_active_users"))
    )
    .withColumn(
        "m365_active_users",
        F.when(F.col("m365_active_users") > F.col("m365_licensed_seats"),
               F.col("m365_licensed_seats"))
         .otherwise(F.col("m365_active_users"))
    )
    .withColumn(
        "teams_daily_active",
        F.when(F.col("teams_daily_active") < 0, 0)
         .when(F.col("teams_daily_active") > F.col("m365_active_users"),
               F.col("m365_active_users"))
         .otherwise(F.col("teams_daily_active"))
    )
    .withColumn(
        "dynamics_users",
        F.when(F.col("dynamics_users") < 0, 0)
         .when(F.col("dynamics_users") > F.col("m365_active_users"),
               F.col("m365_active_users"))
         .otherwise(F.col("dynamics_users"))
    )
    .withColumn(
        "support_tickets",
        F.when(F.col("support_tickets") < 0, 0)
         .otherwise(F.col("support_tickets"))
    )
)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 16, Finished, Available, Finished, False)

In [16]:
# Silver feature engineering

silver_df = (
    silver_df
    .withColumn(
        "active_user_rate",
        F.when(F.col("m365_licensed_seats") > 0,
               F.round(F.col("m365_active_users") / F.col("m365_licensed_seats"), 4))
         .otherwise(F.lit(0.0))
    )
    .withColumn("m365_active_flag", F.when(F.col("m365_active_users") > 0, 1).otherwise(0))
    .withColumn("azure_active_flag", F.when(F.col("azure_compute_hours") > 20, 1).otherwise(0))
    .withColumn("dynamics_active_flag", F.when(F.col("dynamics_users") > 0, 1).otherwise(0))
    .withColumn(
        "product_breadth_score",
        F.col("m365_active_flag") +
        F.col("azure_active_flag") +
        F.col("dynamics_active_flag") +
        F.col("security_product")
    )
    .withColumn(
        "support_spike_flag",
        F.when(F.col("support_tickets") > 3, 1).otherwise(0)
    )
)

display(
    silver_df.select(
        "active_user_rate",
        "product_breadth_score",
        "support_spike_flag",
        "azure_active_flag"
    ).limit(20)
)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ba954cf2-f5e7-4b05-883e-141b42484a59)

In [23]:
# Seat growth metrics

w = Window.partitionBy("account_id").orderBy("month")

silver_df = (
    silver_df
    .withColumn("prev_licensed_seats", F.lag("m365_licensed_seats").over(w))
    .withColumn(
        "seat_growth_mom",
        F.when(F.col("prev_licensed_seats").isNull(), None)
         .otherwise(F.col("m365_licensed_seats") - F.col("prev_licensed_seats"))
    )
    .withColumn(
        "no_growth_flag",
        F.when(F.col("is_pre_signup") == 1, 0)
         .when(F.col("prev_licensed_seats").isNull(), 0)
         .when(F.col("m365_licensed_seats") <= F.col("prev_licensed_seats"), 1)
         .otherwise(0)
    )
    .withColumn(
        "growth_reset_flag",
        F.when(F.col("no_growth_flag") == 0, 1).otherwise(0)
    )
)

w2 = Window.partitionBy("account_id").orderBy("month").rowsBetween(Window.unboundedPreceding, 0)

silver_df = silver_df.withColumn(
    "growth_group",
    F.sum("growth_reset_flag").over(w2)
)

w3 = Window.partitionBy("account_id", "growth_group").orderBy("month")

silver_df = silver_df.withColumn(
    "months_without_growth",
    F.when(F.col("no_growth_flag") == 1, F.row_number().over(w3) - 1)
     .otherwise(0)
)

display(
    silver_df.select(
        "account_id", "signup_month", "month", "is_pre_signup",
        "m365_licensed_seats", "prev_licensed_seats",
        "seat_growth_mom", "no_growth_flag", "months_without_growth"
    ).limit(40)
)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9b36a4e-2e04-4f15-9d81-9af8d6814be7)

In [24]:
# Final Silver validation

validation_df = silver_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("account_id").alias("distinct_accounts"),
    F.sum((F.col("m365_active_users") > F.col("m365_licensed_seats")).cast("int")).alias("bad_active_vs_seats"),
    F.sum((F.col("teams_daily_active") > F.col("m365_active_users")).cast("int")).alias("bad_teams_vs_active"),
    F.sum((F.col("dynamics_users") > F.col("m365_active_users")).cast("int")).alias("bad_dynamics_vs_active"),
    F.sum((F.col("product_breadth_score") < 0).cast("int")).alias("bad_breadth_low"),
    F.sum((F.col("product_breadth_score") > 4).cast("int")).alias("bad_breadth_high"),
    F.sum((F.col("azure_compute_hours") < 0).cast("int")).alias("negative_azure_remaining"),
    F.sum((F.col("support_tickets") == 99).cast("int")).alias("support_99_remaining"),
    F.sum((F.col("active_user_rate") < 0).cast("int")).alias("bad_active_rate_low"),
    F.sum((F.col("active_user_rate") > 1).cast("int")).alias("bad_active_rate_high")
)

display(validation_df)

display(silver_df.select("industry").distinct().orderBy("industry"))
display(silver_df.select("revenue_tier").distinct().orderBy("revenue_tier"))
display(
    silver_df.select(
        "account_id", "month", "industry", "revenue_tier",
        "m365_active_users", "m365_licensed_seats",
        "active_user_rate", "product_breadth_score",
        "support_spike_flag", "months_without_growth"
    ).limit(20)
)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e4fef56a-3d6b-426d-b56c-dd2b625d4388)

SynapseWidget(Synapse.DataFrame, 04143665-d114-4a97-a50c-41ee7f84a30c)

SynapseWidget(Synapse.DataFrame, aa43e9c4-3872-4cba-a468-0596e0b064cc)

SynapseWidget(Synapse.DataFrame, d93e885e-6a1e-493e-9b17-b85cc1981364)

In [25]:
# Final Silver column selection

silver_final = silver_df.select(
    "account_id",
    "company_name",
    "signup_month",
    "month",
    "snapshot_date",
    "industry",
    "employee_count",
    "revenue_tier",
    "region",
    "m365_active_users",
    "m365_licensed_seats",
    "teams_daily_active",
    "azure_compute_hours",
    "dynamics_users",
    "security_product",
    "support_tickets",
    "estimated_clv_12mo",
    "churned",
    "is_pre_signup",
    "active_user_rate",
    "m365_active_flag",
    "azure_active_flag",
    "dynamics_active_flag",
    "product_breadth_score",
    "support_spike_flag",
    "seat_growth_mom",
    "months_without_growth"
)

print(f"Final Silver rows   : {silver_final.count():,}")
print(f"Final Silver columns: {len(silver_final.columns)}")
display(silver_final.limit(10))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 27, Finished, Available, Finished, False)

Final Silver rows   : 58,500
Final Silver columns: 27


SynapseWidget(Synapse.DataFrame, 4d53f906-7241-4317-b11f-164f35191784)

In [28]:
display(
    silver_df.filter(F.col("account_id").isin(1, 2, 3))
    .select(
        "account_id", "signup_month", "month", "is_pre_signup",
        "m365_licensed_seats", "prev_licensed_seats",
        "seat_growth_mom", "no_growth_flag", "months_without_growth"
    )
    .orderBy("account_id", "month")
)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9a82c42b-b02f-419b-a757-b0224aba9abc)

In [29]:
display(spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT account_id) AS distinct_accounts,
    SUM(CASE WHEN m365_active_users > m365_licensed_seats THEN 1 ELSE 0 END) AS bad_active_vs_seats,
    SUM(CASE WHEN teams_daily_active > m365_active_users THEN 1 ELSE 0 END) AS bad_teams_vs_active,
    SUM(CASE WHEN dynamics_users > m365_active_users THEN 1 ELSE 0 END) AS bad_dynamics_vs_active,
    SUM(CASE WHEN product_breadth_score < 0 OR product_breadth_score > 4 THEN 1 ELSE 0 END) AS bad_breadth,
    SUM(CASE WHEN azure_compute_hours < 0 THEN 1 ELSE 0 END) AS negative_azure_remaining,
    SUM(CASE WHEN support_tickets = 99 THEN 1 ELSE 0 END) AS support_99_remaining,
    SUM(CASE WHEN active_user_rate < 0 OR active_user_rate > 1 THEN 1 ELSE 0 END) AS bad_active_user_rate
FROM silver_smb_accounts
"""))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a45b7292-8cb5-4970-bea1-6a544f704a71)

In [30]:
display(spark.sql("""
SELECT
    COUNT(*) AS quarantine_rows
FROM silver_smb_accounts_quarantine
"""))

display(spark.sql("""
SELECT DISTINCT industry
FROM silver_smb_accounts
ORDER BY industry
"""))

display(spark.sql("""
SELECT DISTINCT revenue_tier
FROM silver_smb_accounts
ORDER BY revenue_tier
"""))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc93262a-eda8-4294-8beb-41224f40fb41)

SynapseWidget(Synapse.DataFrame, 7b924068-579d-45aa-b5d2-749c2b667fb9)

SynapseWidget(Synapse.DataFrame, f82f35b9-97e8-47d1-91b1-c42ab00fc075)

In [31]:
# Save Silver table

silver_final.write.mode("overwrite").saveAsTable(SILVER_TABLE)

saved_df = spark.table(SILVER_TABLE)

print(f"{SILVER_TABLE} saved successfully")
print(f"Rows   : {saved_df.count():,}")
print(f"Columns: {len(saved_df.columns)}")

display(saved_df.limit(5))

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 33, Finished, Available, Finished, False)

silver_smb_accounts saved successfully
Rows   : 58,500
Columns: 27


SynapseWidget(Synapse.DataFrame, 758f285f-a90e-4730-b901-209ed2a6714f)

In [36]:
quarantine_df.write.mode("overwrite").saveAsTable(QUARANTINE_TABLE)

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 38, Finished, Available, Finished, False)

In [37]:
# Silver summary

print("Silver layer complete.")
print("- Standardized month and categorical fields")
print("- Quarantined unrecoverable records")
print("- Repaired sentinel values and invalid telemetry")
print("- Enforced logical usage constraints")
print("- Engineered monthly account health features")
print("- Saved analytics-ready Silver table: silver_smb_accounts")

StatementMeta(, b5a11774-8ab4-4029-a005-eabf8a015b86, 39, Finished, Available, Finished, False)

Silver layer complete.
- Standardized month and categorical fields
- Quarantined unrecoverable records
- Repaired sentinel values and invalid telemetry
- Enforced logical usage constraints
- Engineered monthly account health features
- Saved analytics-ready Silver table: silver_smb_accounts
